# 🗿 modelo3d — De foto a modelo 3D imprimible

Convertí una foto (o tres vistas del mismo objeto) en un archivo **STL listo para imprimir**, sin saber nada de programación.

## Qué necesitás
- Una cuenta de Google (gratis).
- Una foto del objeto: buena luz, fondo liso, un solo objeto centrado.

## Cuánto tarda
- **Primera vez:** 5–8 minutos de instalación automática (solo una vez por sesión).
- **Cada modelo:** entre 30 segundos y 2 minutos.

## Antes de empezar
1. Hacé clic en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
2. Ejecutá la celda de instalación de abajo y esperá el mensaje ✅.
3. La aplicación va a aparecer al final de la página.

⚠️ **Importante:** cuando la sesión de Colab se cierre, los archivos se borran. Descargá tu STL apenas lo generes.


In [ ]:
# --- Configuración general ---
SIZE_PRESETS_MM = {"10cm": 100, "15cm": 150}
DEFAULT_PRESET = "10cm"


def resolve_size_preset(preset: str, custom_mm: int | None) -> int:
    """Devuelve la altura objetivo en milímetros."""
    if preset in SIZE_PRESETS_MM:
        return SIZE_PRESETS_MM[preset]
    if preset == "custom":
        if not custom_mm or custom_mm <= 0:
            raise ValueError("Ingresá un alto en milímetros válido (mayor a 0).")
        return int(custom_mm)
    raise ValueError(f"Tamaño desconocido: {preset}")

In [ ]:
# --- Validación de fotos y mensajes ---
import numpy as np
from PIL import Image

MIN_SIDE_PX = 256

ERRORS_ES = {
    "too_small": "La foto es muy chica. Usá una imagen de al menos 256 píxeles por lado.",
    "unreadable": "No pudimos leer la imagen. Probá con otro archivo JPG o PNG.",
    "no_object": "No detectamos ningún objeto en la foto. Revisá que el objeto se vea completo y con buen contraste contra el fondo.",
    "bad_cutout": "El recorte del objeto quedó raro. Sacá la foto con el objeto centrado sobre un fondo liso, sin manos y sin que se corte con el borde.",
}

FALLBACK_ERROR_ES = (
    "Algo salió mal generando el modelo. Probá de nuevo; si sigue fallando, "
    "probá con otra foto."
)


def _to_rgb(img: Image.Image) -> Image.Image:
    return img.convert("RGB") if img.mode != "RGB" else img


def validate_image(img: Image.Image) -> None:
    try:
        img = _to_rgb(img)
        w, h = img.size
    except Exception as exc:
        raise ValueError(ERRORS_ES["unreadable"]) from exc
    if w < MIN_SIDE_PX or h < MIN_SIDE_PX:
        raise ValueError(ERRORS_ES["too_small"])


def mask_fraction(mask: np.ndarray) -> float:
    return float(np.count_nonzero(mask)) / float(mask.size)


def check_mask_sane(fraction: float) -> None:
    if fraction < 0.01:
        raise ValueError(ERRORS_ES["no_object"])
    if fraction > 0.90:
        raise ValueError(ERRORS_ES["bad_cutout"])


def friendly_error(exc: Exception) -> str:
    if isinstance(exc, ValueError) and str(exc) in ERRORS_ES.values():
        return str(exc)
    return FALLBACK_ERROR_ES
